# RQ2 Main Model: Standard Binomial Mixed-Effects Logistic (Task Random Intercept Only)

This notebook fits a frequentist binomial mixed-effects logistic model using R `lme4::glmer`:

`Defense ~ Harness * Model + AttackCategory + (1 | Task)`

- `Defense=1`：TAF；`Defense=0`：TAS/FAS。
- Harness, Model, and AttackCategory are specified as fixed effects.
- Task enters as a random intercept to account for repeated measurements of the same task across harness/model combinations, as well as inherent task difficulty.
- Per requirements, this version does not include `(1 | PageLayout)`。
- Each task maps to exactly one webpage, so the Task random intercept also absorbs the idiosyncratic difficulty of the task’s corresponding page.
- Only conditional DSR is analyzed in the current version; capability-dependent missingness due to FAF or empty flags is not addressed.

In [ ]:
import getpass
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import pymysql
from IPython.display import display

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

## 1. Environment and Experimental Scope

The notebook kernel should be set to `Python 3.10 (py310)`. R dependencies are installed in the same Conda environment. MySQL credentials are read from environment variables or interactive input, and are not stored in the notebook.

In [2]:
HARNESSES = [
    'seeact', 'browser_text', 'browser_vision',
    'agent_e', 'skyvern_1', 'skyvern_2',
]
MODELS = [
    'claude_3_7', 'claude_4_5', 'gpt_4o',
    'gpt_5_1', 'gpt_5_2', 'qwen_235b', 'qwen_30b',
]
ATTACK_CATEGORIES = ['mup', 'mpi', 'dwd']

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'benchmark/skyvern-1.0/all_task_list.json').exists():
            return candidate
    raise FileNotFoundError('Cannot locate repository root')

REPO_ROOT = find_repo_root()
TASK_METADATA_PATH = REPO_ROOT / 'benchmark/skyvern-1.0/all_task_list.json'
OUTPUT_DIR = REPO_ROOT / 'web-agent/harness_security/analysis_task_only_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_CSV = OUTPUT_DIR / 'main_model_input.csv'

default_rscript = Path('/opt/homebrew/anaconda3/envs/py310/bin/Rscript')
RSCRIPT = Path(os.getenv('PY310_RSCRIPT', str(default_rscript)))
if not RSCRIPT.exists():
    raise FileNotFoundError(f'Rscript not found: {RSCRIPT}')

version_check = subprocess.run(
    [str(RSCRIPT), '-e',
     'cat(R.version.string, \"\n\"); library(lme4); library(emmeans); library(car); library(broom.mixed); cat(\"R packages OK\n\")'],
    text=True, capture_output=True, check=True,
)
print(version_check.stdout)
print('Repository root:', REPO_ROOT)
print('Output directory:', OUTPUT_DIR)

R version 4.3.3 (2024-02-29) 
R packages OK

Repository root: /Users/janx/Desktop/Security_Lab/llm
Output directory: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_task_only_output


## 2. Verify Task–Webpage One-to-One Mapping

If Task and Webpage maintain a strict one-to-one correspondence, the data cannot support separate estimation of `(1|Task)` and `(1|Webpage)`; only `(1|Task)` is retained in this model.

In [3]:
task_meta = pd.DataFrame(json.loads(TASK_METADATA_PATH.read_text(encoding='utf-8')))
mapping_audit = pd.Series({
    'metadata_rows': len(task_meta),
    'unique_tasks': task_meta['task_id'].nunique(),
    'unique_webpages': task_meta['website'].nunique(),
    'max_webpages_per_task': task_meta.groupby('task_id')['website'].nunique().max(),
    'max_tasks_per_webpage': task_meta.groupby('website')['task_id'].nunique().max(),
}, name='value').to_frame()
display(mapping_audit)
assert len(task_meta) == task_meta['task_id'].nunique()
assert len(task_meta) == task_meta['website'].nunique()

,value
metadata_rows,618
unique_tasks,618
unique_webpages,618
max_webpages_per_task,1
max_tasks_per_webpage,1


## 3. Load and Audit 6 × 7 Experimental Units

In [ ]:
MYSQL_PASSWORD = os.getenv('HARNESS_MYSQL_PASSWORD')
if MYSQL_PASSWORD is None:
    MYSQL_PASSWORD = getpass.getpass('MySQL password for user agentguard: ')

MYSQL_CONFIG = {
    'host': os.getenv('HARNESS_MYSQL_HOST', ''),
    'port': int(os.getenv('HARNESS_MYSQL_PORT', '')),
    'user': os.getenv('HARNESS_MYSQL_USER', ''),
    'password': MYSQL_PASSWORD,
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor,
}

def quote_identifier(name):
    return '`' + str(name).replace('`', '``') + '`'

def load_experiment_rows():
    records = []
    conn = pymysql.connect(**MYSQL_CONFIG)
    try:
        with conn.cursor() as cursor:
            for harness in HARNESSES:
                for backend_model in MODELS:
                    table = f'{quote_identifier(harness)}.{quote_identifier(backend_model)}'
                    cursor.execute(f'''
                        SELECT task_id, LOWER(TRIM(flag)) AS flag,
                               LOWER(TRIM(benchmark)) AS attack_category
                        FROM {table}
                    ''')
                    records.extend({
                        'harness': harness,
                        'backend_model': backend_model,
                        **row,
                    } for row in cursor.fetchall())
    finally:
        conn.close()
    return pd.DataFrame(records)

raw_df = load_experiment_rows()
raw_df = raw_df.merge(
    task_meta[['task_id', 'website']],
    on='task_id', how='left', validate='many_to_one', indicator=True,
)
assert raw_df['_merge'].eq('both').all()
raw_df = raw_df.drop(columns='_merge')

cell_audit = (
    raw_df.groupby(['harness', 'backend_model'])
    .agg(rows=('task_id', 'size'), unique_tasks=('task_id', 'nunique'))
    .reset_index()
)
display(cell_audit)
assert cell_audit['rows'].eq(618).all()
assert cell_audit['unique_tasks'].eq(618).all()
assert not raw_df.duplicated(['harness', 'backend_model', 'task_id']).any()
print(f'Raw rows: {len(raw_df):,}')

,harness,backend_model,rows,unique_tasks
0,agent_e,claude_3_7,618,618
1,agent_e,claude_4_5,618,618
2,agent_e,gpt_4o,618,618
3,agent_e,gpt_5_1,618,618
4,agent_e,gpt_5_2,618,618
5,agent_e,qwen_235b,618,618
6,agent_e,qwen_30b,618,618
7,browser_text,claude_3_7,618,618
8,browser_text,claude_4_5,618,618
9,browser_text,gpt_4o,618,618


Raw rows: 25,956


## 4. Construct Conditional DSR Main Model Dataset

FAF and empty flags are first quantified for reporting purposes, but are excluded from the current Defense model.

In [5]:
flag_by_cell = pd.crosstab(
    [raw_df['harness'], raw_df['backend_model']],
    raw_df['flag'].fillna('missing'),
).reset_index()
display(flag_by_cell)

analysis_df = raw_df[raw_df['flag'].isin(['tas', 'fas', 'taf'])].copy()
analysis_df = analysis_df[analysis_df['attack_category'].isin(ATTACK_CATEGORIES)].copy()
analysis_df['defense'] = analysis_df['flag'].eq('taf').astype(int)

analysis_audit = pd.Series({
    'valid_rows': len(analysis_df),
    'successes': int(analysis_df['defense'].sum()),
    'failures': int((1 - analysis_df['defense']).sum()),
    'unique_tasks_with_valid_outcome': analysis_df['task_id'].nunique(),
    'harnesses': analysis_df['harness'].nunique(),
    'models': analysis_df['backend_model'].nunique(),
    'attack_categories': analysis_df['attack_category'].nunique(),
}, name='value').to_frame()
display(analysis_audit)
display(
    analysis_df.groupby(['harness', 'backend_model'])
    .agg(n=('defense', 'size'), raw_dsr=('defense', 'mean'))
    .reset_index()
)

r_input = analysis_df[[
    'defense', 'harness', 'backend_model', 'attack_category', 'task_id'
]].copy()
r_input.to_csv(INPUT_CSV, index=False)
print('Saved model input:', INPUT_CSV)

flag,harness,backend_model,faf,fas,missing,taf,tas
0,agent_e,claude_3_7,148,44,92,304,30
1,agent_e,claude_4_5,127,74,92,300,25
2,agent_e,gpt_4o,199,29,92,274,24
3,agent_e,gpt_5_1,174,35,92,296,21
4,agent_e,gpt_5_2,236,36,92,221,33
5,agent_e,qwen_235b,257,27,92,221,21
6,agent_e,qwen_30b,283,25,92,185,33
7,browser_text,claude_3_7,174,72,92,238,42
8,browser_text,claude_4_5,90,91,92,319,26
9,browser_text,gpt_4o,104,107,92,257,58


,value
valid_rows,15783
successes,10646
failures,5137
unique_tasks_with_valid_outcome,522
harnesses,6
models,7
attack_categories,3


,harness,backend_model,n,raw_dsr
0,agent_e,claude_3_7,378,0.804233
1,agent_e,claude_4_5,399,0.751880
2,agent_e,gpt_4o,327,0.837920
3,agent_e,gpt_5_1,352,0.840909
4,agent_e,gpt_5_2,290,0.762069
5,agent_e,qwen_235b,269,0.821561
6,agent_e,qwen_30b,243,0.761317
7,browser_text,claude_3_7,352,0.676136
8,browser_text,claude_4_5,436,0.731651
9,browser_text,gpt_4o,422,0.609005


Saved model input: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_task_only_output/main_model_input.csv


## 5. Fit Standard Frequentist GLMM Using lme4::glmer

Sum-to-zero contrasts are specified so that Type III main-effect tests in the presence of interactions represent overall effects averaged across the other factor. The model uses the Laplace approximation (`nAGQ=1`) and the `bobyqa` optimizer.

In [6]:
R_ANALYSIS = r'''
args <- commandArgs(trailingOnly=TRUE)
input_csv <- args[[1]]
output_dir <- args[[2]]

suppressPackageStartupMessages({
  library(lme4)
  library(emmeans)
  library(car)
  library(broom.mixed)
})

options(contrasts=c('contr.sum', 'contr.poly'))
d <- read.csv(input_csv, stringsAsFactors=FALSE)
d$harness <- factor(d$harness, levels=c(
  'seeact','browser_text','browser_vision','agent_e','skyvern_1','skyvern_2'))
d$backend_model <- factor(d$backend_model, levels=c(
  'claude_3_7','claude_4_5','gpt_4o','gpt_5_1','gpt_5_2','qwen_235b','qwen_30b'))
d$attack_category <- factor(d$attack_category, levels=c('mup','mpi','dwd'))
d$task_id <- factor(d$task_id)

fit <- glmer(
  defense ~ harness * backend_model + attack_category + (1 | task_id),
  data=d, family=binomial(link='logit'), nAGQ=1,
  control=glmerControl(
    optimizer='bobyqa', optCtrl=list(maxfun=200000), calc.derivs=TRUE)
)
saveRDS(fit, file.path(output_dir, 'main_model_glmer.rds'))
capture.output(summary(fit), file=file.path(output_dir, 'model_summary.txt'))

conv_messages <- fit@optinfo$conv$lme4$messages
convergence <- data.frame(
  converged=is.null(conv_messages),
  singular=isSingular(fit, tol=1e-4),
  optimizer=fit@optinfo$optimizer,
  message=if (is.null(conv_messages)) 'none' else paste(conv_messages, collapse='; '),
  n=nobs(fit), logLik=as.numeric(logLik(fit)), AIC=AIC(fit), BIC=BIC(fit)
)
write.csv(convergence, file.path(output_dir, 'convergence.csv'), row.names=FALSE)

coef_table <- as.data.frame(summary(fit)$coefficients)
coef_table$term <- rownames(coef_table)
rownames(coef_table) <- NULL
names(coef_table)[1:4] <- c('log_odds','std_error','z_value','p_value')
ci <- confint(fit, parm='beta_', method='Wald')
coef_table$log_odds_low_95 <- ci[,1]
coef_table$log_odds_high_95 <- ci[,2]
coef_table$odds_ratio <- exp(coef_table$log_odds)
coef_table$or_low_95 <- exp(coef_table$log_odds_low_95)
coef_table$or_high_95 <- exp(coef_table$log_odds_high_95)
coef_table <- coef_table[,c(
  'term','log_odds','std_error','z_value','p_value',
  'log_odds_low_95','log_odds_high_95','odds_ratio','or_low_95','or_high_95')]
write.csv(coef_table, file.path(output_dir, 'fixed_effects.csv'), row.names=FALSE)

omnibus <- as.data.frame(Anova(fit, type=3, test.statistic='Chisq'))
omnibus$source <- rownames(omnibus)
rownames(omnibus) <- NULL
omnibus <- omnibus[,c('source', setdiff(names(omnibus), 'source'))]
write.csv(omnibus, file.path(output_dir, 'omnibus_type3.csv'), row.names=FALSE)

random_effects <- as.data.frame(VarCorr(fit))
task_variance <- random_effects$vcov[random_effects$grp == 'task_id'][1]
random_effects$latent_icc <- ifelse(
  random_effects$grp == 'task_id', task_variance/(task_variance + pi^2/3), NA)
write.csv(random_effects, file.path(output_dir, 'random_effects.csv'), row.names=FALSE)

write_emm <- function(spec, filename) {
  obj <- emmeans(fit, spec, type='response', weights='equal')
  out <- as.data.frame(summary(obj, infer=c(TRUE, TRUE)))
  write.csv(out, file.path(output_dir, filename), row.names=FALSE)
  invisible(obj)
}
emm_h <- write_emm(~ harness, 'emm_harness.csv')
emm_m <- write_emm(~ backend_model, 'emm_model.csv')
emm_a <- write_emm(~ attack_category, 'emm_attack_category.csv')
emm_config <- write_emm(~ harness * backend_model, 'emm_configurations.csv')

h_pairs <- as.data.frame(summary(
  pairs(emmeans(fit, ~ harness, weights='equal'), adjust='tukey'),
  infer=c(TRUE, TRUE), type='response'))
m_pairs <- as.data.frame(summary(
  pairs(emmeans(fit, ~ backend_model, weights='equal'), adjust='tukey'),
  infer=c(TRUE, TRUE), type='response'))
write.csv(h_pairs, file.path(output_dir, 'pairwise_harness_tukey.csv'), row.names=FALSE)
write.csv(m_pairs, file.path(output_dir, 'pairwise_model_tukey.csv'), row.names=FALSE)

cat('Model fitted successfully.\n')
print(convergence)
print(omnibus)
'''

run = subprocess.run(
    [str(RSCRIPT), '-e', R_ANALYSIS, str(INPUT_CSV), str(OUTPUT_DIR)],
    text=True, capture_output=True,
)
print(run.stdout)
if run.stderr:
    print('R messages/warnings:\n', run.stderr)
if run.returncode != 0:
    raise RuntimeError(f'R model failed with exit code {run.returncode}')

Model fitted successfully.
  converged singular optimizer message     n   logLik      AIC      BIC
1      TRUE    FALSE    bobyqa    none 15783 -5497.37 11084.74 11429.74
                 source      Chisq Df   Pr(>Chisq)
1           (Intercept)  75.212291  1 4.227273e-18
2               harness 331.173121  5 1.974696e-69
3         backend_model 437.632057  6 2.251666e-91
4       attack_category   4.812647  2 9.014609e-02
5 harness:backend_model 299.813178 30 2.882264e-46

R messages/warnings:
 
Correlation matrix not shown by default, as p = 44 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it

NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions



## 6. Convergence, Random Effects, and Omnibus Tests

Type III Wald χ² tests are computed using sum contrasts. In the presence of the Harness × Model interaction, main effects reflect overall effects averaged across the levels of the other factor; final interpretation must nonetheless be informed by configuration-specific probabilities and simple effects.

In [7]:
convergence = pd.read_csv(OUTPUT_DIR / 'convergence.csv')
random_effects = pd.read_csv(OUTPUT_DIR / 'random_effects.csv')
omnibus = pd.read_csv(OUTPUT_DIR / 'omnibus_type3.csv')
display(convergence)
display(random_effects)
display(omnibus)
assert bool(convergence.loc[0, 'converged']), convergence.loc[0, 'message']
assert not bool(convergence.loc[0, 'singular']), 'Model has a singular random-effects fit'

,converged,singular,optimizer,message,n,logLik,AIC,BIC
0,True,False,bobyqa,none,15783,-5497.370252,11084.740505,11429.741496


,grp,var1,var2,vcov,sdcor,latent_icc
0,task_id,(Intercept),NaN,9.94603,3.153733,0.751444


,source,Chisq,Df,Pr(>Chisq)
0,(Intercept),75.212291,1,4.227273e-18
1,harness,331.173121,5,1.974696e-69
2,backend_model,437.632057,6,2.251666e-91
3,attack_category,4.812647,2,9.014609e-02
4,harness:backend_model,299.813178,30,2.882264e-46


## 7. Fixed Effects, Estimated Marginal Means, and 95% CIs

`emmeans` are equally weighted averages over the fixed factors not displayed, such that Harness/Model probabilities are not driven by imbalanced proportions of attack categories in the available sample. The configuration table is averaged equally across AttackCategory. Note that these response‑scale probabilities represent fixed‑effect / typical‑task predictions with the Task random effect set to zero, and are not integrated over the distribution of the Task random effect. For this reason, they should not be directly equated to raw pooled DSR; omnibus tests and odds‑ratio contrasts are unaffected by this presentation scale.

In [8]:
fixed_effects = pd.read_csv(OUTPUT_DIR / 'fixed_effects.csv')
emm_harness = pd.read_csv(OUTPUT_DIR / 'emm_harness.csv')
emm_model = pd.read_csv(OUTPUT_DIR / 'emm_model.csv')
emm_attack = pd.read_csv(OUTPUT_DIR / 'emm_attack_category.csv')
emm_config = pd.read_csv(OUTPUT_DIR / 'emm_configurations.csv')
display(fixed_effects)
display(emm_harness)
display(emm_model)
display(emm_attack)
display(emm_config)

,term,log_odds,std_error,z_value,p_value,log_odds_low_95,log_odds_high_95,odds_ratio,or_low_95,or_high_95
0,(Intercept),1.262101,0.145529,8.672502,4.227273e-18,0.976870,1.547333,3.532837,2.656128,4.698922
1,harness1,-0.222903,0.070285,-3.171407,1.517026e-03,-0.360659,-0.085146,0.800193,0.697217,0.918378
2,harness2,-0.029482,0.057323,-0.514314,6.070327e-01,-0.141832,0.082869,0.970948,0.867767,1.086399
3,harness3,-0.185572,0.053762,-3.451732,5.570013e-04,-0.290944,-0.080200,0.830629,0.747558,0.922931
4,harness4,0.901929,0.072816,12.386412,3.095889e-35,0.759212,1.044645,2.464351,2.136592,2.842390
5,harness5,0.305920,0.062925,4.861624,1.164264e-06,0.182588,0.429251,1.357873,1.200320,1.536107
6,backend_model1,-0.079927,0.064999,-1.229675,2.188189e-01,-0.207322,0.047468,0.923184,0.812758,1.048612
7,backend_model2,1.135727,0.076012,14.941419,1.771622e-50,0.986746,1.284707,3.113435,2.682491,3.613610
8,backend_model3,-0.489202,0.063013,-7.763556,8.258059e-15,-0.612704,-0.365699,0.613116,0.541884,0.693711
9,backend_model4,0.411610,0.070143,5.868198,4.405577e-09,0.274134,0.549087,1.509246,1.315390,1.731672


,harness,prob,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value
0,seeact,0.738695,0.031445,inf,0.672587,0.795514,0.5,6.379124,1.781036e-10
1,browser_text,0.774277,0.027227,inf,0.716524,0.823167,0.5,7.912277,2.527229e-15
2,browser_vision,0.745837,0.029141,inf,0.684652,0.798641,0.5,7.002967,2.505987e-12
3,agent_e,0.896973,0.015266,inf,0.862980,0.923282,0.5,13.100324,3.278558e-39
4,skyvern_1,0.827501,0.022707,inf,0.778377,0.867589,0.5,9.856888,6.400251e-23
5,skyvern_2,0.620627,0.035904,inf,0.548183,0.688065,0.5,3.227794,1.247488e-03


,backend_model,prob,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value
0,claude_3_7,0.765339,0.028536,inf,0.704904,0.816616,0.5,7.440295,1.004609e-13
1,claude_4_5,0.916662,0.012655,inf,0.888265,0.938342,0.5,14.474306,1.760934e-47
2,gpt_4o,0.684148,0.034075,inf,0.613924,0.746865,0.5,4.901434,9.513948e-07
3,gpt_5_1,0.842070,0.021599,inf,0.795010,0.879958,0.5,10.305432,6.659196e-25
4,gpt_5_2,0.823256,0.023641,inf,0.772085,0.864948,0.5,9.469595,2.809314e-21
5,qwen_235b,0.693557,0.033504,inf,0.624299,0.755057,0.5,5.181490,2.201199e-07
6,qwen_30b,0.611287,0.037458,inf,0.535877,0.681722,0.5,2.871861,4.080618e-03


,attack_category,prob,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value
0,mup,0.804250,0.041196,inf,0.710990,0.872800,0.5,5.400086,6.660879e-08
1,mpi,0.693583,0.055943,inf,0.574687,0.791311,0.5,3.103447,1.912807e-03
2,dwd,0.825823,0.032853,inf,0.751877,0.881213,0.5,6.813867,9.500970e-12


,harness,backend_model,prob,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value
0,seeact,claude_3_7,0.652404,0.055845,inf,0.536675,0.752554,0.5,2.556747,1.056560e-02
1,browser_text,claude_3_7,0.743405,0.041888,inf,0.653249,0.816698,0.5,4.844185,1.271324e-06
2,browser_vision,claude_3_7,0.763508,0.036993,inf,0.683621,0.828290,0.5,5.720608,1.061433e-08
3,agent_e,claude_3_7,0.918450,0.018315,inf,0.874596,0.947883,0.5,9.902854,4.045613e-23
4,skyvern_1,claude_3_7,0.843994,0.029643,inf,0.776781,0.893738,0.5,7.498973,6.431992e-14
5,skyvern_2,claude_3_7,0.529451,0.047740,inf,0.435944,0.620935,0.5,0.615472,5.382430e-01
6,seeact,claude_4_5,0.941372,0.015216,inf,0.903417,0.964990,0.5,10.069722,7.519031e-24
7,browser_text,claude_4_5,0.871018,0.024740,inf,0.814327,0.912265,0.5,8.673301,4.197710e-18
8,browser_vision,claude_4_5,0.883851,0.022363,inf,0.832363,0.921026,0.5,9.316283,1.204870e-20
9,agent_e,claude_4_5,0.891456,0.022827,inf,0.837987,0.928779,0.5,8.925778,4.425784e-19


## 8. Tukey-Adjusted Pairwise Comparisons of Harness and Model

These are overall comparisons averaged across the other factor and AttackCategory. Given the potential for substantial interaction effects, configuration‑level simple effects for the paper require further elaboration during the subsequent interpretation phase.

In [9]:
pairwise_harness = pd.read_csv(OUTPUT_DIR / 'pairwise_harness_tukey.csv')
pairwise_model = pd.read_csv(OUTPUT_DIR / 'pairwise_model_tukey.csv')
display(pairwise_harness)
display(pairwise_model)

,contrast,odds.ratio,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value
0,seeact / browser_text,0.824135,0.082593,inf,0.619394,1.096553,1,-1.929996,3.836306e-01
1,seeact / browser_vision,0.963357,0.093777,inf,0.729984,1.271339,1,-0.383494,9.989292e-01
2,seeact / agent_e,0.324707,0.036764,inf,0.235162,0.448349,1,-9.934808,5.229150e-14
3,seeact / skyvern_1,0.589298,0.062282,inf,0.436049,0.796408,1,-5.003637,8.345545e-06
4,seeact / skyvern_2,1.728042,0.167050,inf,1.311941,2.276116,1,5.658305,2.285864e-07
5,browser_text / browser_vision,1.168931,0.096622,inf,0.923614,1.479407,1,1.888384,4.092572e-01
6,browser_text / agent_e,0.393998,0.040622,inf,0.293693,0.528560,1,-9.033861,3.874678e-14
7,browser_text / skyvern_1,0.715051,0.066104,inf,0.549444,0.930573,1,-3.628035,3.877187e-03
8,browser_text / skyvern_2,2.096795,0.173874,inf,1.655497,2.655728,1,8.928785,4.252154e-14
9,browser_vision / agent_e,0.337058,0.033847,inf,0.253178,0.448727,1,-10.829766,5.773160e-14


,contrast,odds.ratio,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value
0,claude_3_7 / claude_4_5,0.296516,0.032318,inf,0.215027,0.408888,1,-11.153713,2.176037e-14
1,claude_3_7 / gpt_4o,1.505725,0.145607,inf,1.132204,2.002473,1,4.232330,4.619029e-04
2,claude_3_7 / gpt_5_1,0.611685,0.063318,inf,0.450800,0.829989,1,-4.748478,4.212205e-05
3,claude_3_7 / gpt_5_2,0.700197,0.072618,inf,0.515732,0.950639,1,-3.436410,1.056701e-02
4,claude_3_7 / qwen_235b,1.441051,0.138906,inf,1.084564,1.914712,1,3.790485,2.865210e-03
5,claude_3_7 / qwen_30b,2.073936,0.200651,inf,1.559245,2.758522,1,7.539604,1.034728e-12
6,claude_4_5 / gpt_4o,5.078054,0.546429,inf,3.697552,6.973974,1,15.100713,0.000000e+00
7,claude_4_5 / gpt_5_1,2.062907,0.231743,inf,1.481281,2.872908,1,6.445860,2.412166e-09
8,claude_4_5 / gpt_5_2,2.361412,0.266608,inf,1.692809,3.294091,1,7.610682,6.155076e-13
9,claude_4_5 / qwen_235b,4.859940,0.523785,inf,3.536961,6.677772,1,14.669551,0.000000e+00


## 9. Interpretation Limitations

1. This is the main conditional DSR model, which has not been adjusted for capability‑dependent missingness.
2. Task and Webpage are in one‑to‑one correspondence, so the Task random intercept captures the combined difficulty of the task and its unique webpage; the two cannot be disentangled from the current design.
3. Per requirements, PageLayout is not included in this version; it can be compared against models with PageLayout as a sensitivity check.
4. When the Harness × Model interaction is significant, one cannot claim that either factor dominates based solely on the p‑values or effect sizes of the two main effects.
5. `confint(..., method='Wald')` and `emmeans` provide frequentist 95% CIs; for a more conservative check, profile‑likelihood or parametric‑bootstrap CIs can be computed separately.